1.0 Ucitavnje Drive-a

In [10]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime

ROOT = Path("/content/drive/MyDrive/Diplomski")
RAW_DIR = ROOT / "Data" / "raw"
CURATED_DIR = ROOT / "Data" / "curated"

RUN_ID = datetime.now().strftime("run_%Y-%m-%d_%H%M%S")
RUN_DIR = ROOT / "Reports" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_DIR:", RAW_DIR)
print("CURATED_DIR:", CURATED_DIR)
print("RUN_DIR:", RUN_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RAW_DIR: /content/drive/MyDrive/Diplomski/Data/raw
CURATED_DIR: /content/drive/MyDrive/Diplomski/Data/curated
RUN_DIR: /content/drive/MyDrive/Diplomski/Reports/run_2026-02-25_121523


1.1 Bootstrap

Ovaj deo priprema okruženje za rad u Colab-u i postavlja zajedničke stvari koje koristimo za sve datasete. Ovde se definiše seed (42) kako bi podela podataka svaki put ispala ista, kao i osnovne konstante (npr. dozvoljene ekstenzije slika).

Takođe se ovde nalaze pomoćne funkcije koje se ponavljaju kod svih dataset-a: pronalazak slika u folderima, stratifikovana podela na train/val/test, čuvanje split fajlova (train.csv, val.csv, test.csv) i kreiranje meta.json. Na kraju, postoje dve glavne funkcije koje rade pripremu: jedna za image datasete (radi preko foldera klasa, a kod SipakMed-a može da koristi CROPPED), i jedna za tabular dataset (uklanja duplikate i pravi stratifikovanu podelu po ciljnoj koloni).

In [11]:
import os, json, random
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
SKIP_IF_EXISTS = True

def list_image_files(base: Path):
    files = []
    for p in base.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            files.append(p)
    return files

def stratified_split(items, labels, train=0.7, val=0.15, test=0.15, seed=42):
    assert abs(train + val + test - 1.0) < 1e-9
    rng = np.random.default_rng(seed)

    by_class = defaultdict(list)
    for x, y in zip(items, labels):
        by_class[y].append(x)

    train_items, val_items, test_items = [], [], []
    for y, xs in by_class.items():
        xs = list(xs)
        rng.shuffle(xs)
        n = len(xs)
        n_train = int(round(n * train))
        n_val = int(round(n * val))
        train_items += [(x, y) for x in xs[:n_train]]
        val_items += [(x, y) for x in xs[n_train:n_train+n_val]]
        test_items += [(x, y) for x in xs[n_train+n_val:]]
    rng.shuffle(train_items)
    rng.shuffle(val_items)
    rng.shuffle(test_items)
    return train_items, val_items, test_items

def save_split_csv(out_dir: Path, split_name: str, pairs):
    df = pd.DataFrame(pairs, columns=["path", "label"])
    df.to_csv(out_dir / f"{split_name}.csv", index=False)
    return df

def prepare_image_dataset(dataset_id: str, root: Path, curated_dir: Path, seed: int = 42,
                          train=0.7, val=0.15, test=0.15, use_subfolder: str | None = None):
    out_dir = curated_dir / dataset_id
    out_dir.mkdir(parents=True, exist_ok=True)

    meta_path = out_dir / "meta.json"
    if SKIP_IF_EXISTS and meta_path.exists():
        return json.load(open(meta_path, "r", encoding="utf-8"))

    class_dirs = [p for p in root.iterdir() if p.is_dir()]
    items, labels = [], []
    missing_sub = []

    for cls_dir in class_dirs:
        base = cls_dir
        if use_subfolder is not None:
            sub = cls_dir / use_subfolder
            if not sub.exists():
                missing_sub.append(cls_dir.name)
                continue
            base = sub

        imgs = list_image_files(base)
        items += [str(p) for p in imgs]
        labels += [cls_dir.name] * len(imgs)

    if len(items) == 0:
        raise RuntimeError(f"{dataset_id}: Nema pronađenih slika. Proveri putanju i strukturu foldera.")

    train_pairs, val_pairs, test_pairs = stratified_split(items, labels, train=train, val=val, test=test, seed=seed)

    df_train = save_split_csv(out_dir, "train", train_pairs)
    df_val = save_split_csv(out_dir, "val", val_pairs)
    df_test = save_split_csv(out_dir, "test", test_pairs)

    meta = {
        "dataset_id": dataset_id,
        "type": "image",
        "root": str(root),
        "seed": seed,
        "split_ratio": {"train": train, "val": val, "test": test},
        "num_images": int(len(items)),
        "class_counts": pd.Series(labels).value_counts().astype(int).to_dict(),
        "splits": {
            "train_rows": int(df_train.shape[0]),
            "val_rows": int(df_val.shape[0]),
            "test_rows": int(df_test.shape[0]),
        }
    }

    if use_subfolder is not None:
        meta["used_subfolder"] = use_subfolder
    if missing_sub:
        meta["missing_subfolder_for_classes"] = missing_sub

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    return meta

def prepare_tabular_thyroid(dataset_id: str, root: Path, curated_dir: Path, seed: int = 42,
                           train=0.7, val=0.15, test=0.15, target_col="Recurred"):
    out_dir = curated_dir / dataset_id
    out_dir.mkdir(parents=True, exist_ok=True)

    meta_path = out_dir / "meta.json"
    if SKIP_IF_EXISTS and meta_path.exists():
        return json.load(open(meta_path, "r", encoding="utf-8"))

    csvs = list(root.glob("*.csv"))
    if len(csvs) == 0:
        raise FileNotFoundError(f"{dataset_id}: Nema CSV fajla u {root}")

    csvs.sort(key=lambda p: p.stat().st_size, reverse=True)
    csv_path = csvs[0]

    df = pd.read_csv(csv_path)

    rows_before = int(df.shape[0])
    dup_rows = int(df.duplicated().sum())
    df_clean = df.drop_duplicates().copy()
    rows_after = int(df_clean.shape[0])

    if target_col not in df_clean.columns:
        raise ValueError(f"{dataset_id}: Ne nalazim target kolonu '{target_col}'. Kolone: {list(df_clean.columns)}")

    missing_total = int(df_clean.isna().sum().sum())
    target_counts = df_clean[target_col].value_counts(dropna=False).to_dict()

    items = list(range(len(df_clean)))
    labels = df_clean[target_col].astype(str).tolist()

    train_pairs, val_pairs, test_pairs = stratified_split(items, labels, train=train, val=val, test=test, seed=seed)

    train_idx = [i for i, _ in train_pairs]
    val_idx = [i for i, _ in val_pairs]
    test_idx = [i for i, _ in test_pairs]

    df_train = df_clean.iloc[train_idx].copy()
    df_val = df_clean.iloc[val_idx].copy()
    df_test = df_clean.iloc[test_idx].copy()

    df_train.to_csv(out_dir / "train.csv", index=False)
    df_val.to_csv(out_dir / "val.csv", index=False)
    df_test.to_csv(out_dir / "test.csv", index=False)

    meta = {
        "dataset_id": dataset_id,
        "type": "tabular",
        "source_csv": str(csv_path),
        "seed": seed,
        "split_ratio": {"train": train, "val": val, "test": test},
        "rows_before": rows_before,
        "duplicate_rows_removed": dup_rows,
        "rows_after": rows_after,
        "columns": list(df_clean.columns),
        "missing_total": missing_total,
        "target_col": target_col,
        "target_counts_after_clean": {k: int(v) for k, v in target_counts.items()},
        "splits": {
            "train_rows": int(df_train.shape[0]),
            "val_rows": int(df_val.shape[0]),
            "test_rows": int(df_test.shape[0]),
            "train_target_counts": df_train[target_col].value_counts().astype(int).to_dict(),
            "val_target_counts": df_val[target_col].value_counts().astype(int).to_dict(),
            "test_target_counts": df_test[target_col].value_counts().astype(int).to_dict(),
        }
    }

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    return meta

1.3 Runner

U prethodnoj verziji notebook-a, priprema je bila rađena odvojeno za svaki dataset: prvo se formirala lista (putanja, klasa), zatim se pravila stratifikovana podela na train/val/test, i na kraju su se čuvali train.csv, val.csv, test.csv i meta.json. To je bilo ispravno, ali je dovodilo do ponavljanja istog koda više puta (za thyroid, lc25000, sipakmed i rm1000), pa je bilo teže održavati konzistentnost ako se promeni seed ili procenat podele.

U ovoj verziji, isti postupak je objedninjen u jednu rutinu. Za svaki dataset se automatski: (1) učitaju podaci iz Data/raw, (2) uradi osnovno čišćenje gde je potrebno (npr. uklanjanje duplih redova kod thyroid), (3) napravi stratifikovana podela 70/15/15 uz seed 42, i (4) sačuvaju split fajlovi i metapodaci u Data/curated/<dataset>/. Kod SipakMed-a se koriste slike iz podfoldera CROPPED radi doslednog ulaza i fokusa na relevantan deo slike.

In [12]:

CURATED_DIR.mkdir(parents=True, exist_ok=True)

results = {}

results["thyroid_recurrence"] = prepare_tabular_thyroid(
    dataset_id="thyroid_recurrence",
    root=RAW_DIR / "thyroid_recurrence",
    curated_dir=CURATED_DIR,
    seed=RANDOM_SEED
)

results["lc25000"] = prepare_image_dataset(
    dataset_id="lc25000",
    root=RAW_DIR / "lc25000",
    curated_dir=CURATED_DIR,
    seed=RANDOM_SEED
)

results["sipakmed"] = prepare_image_dataset(
    dataset_id="sipakmed",
    root=RAW_DIR / "sipakmed",
    curated_dir=CURATED_DIR,
    seed=RANDOM_SEED,
    use_subfolder="CROPPED"
)

results["rm1000_lung_history"] = prepare_image_dataset(
    dataset_id="rm1000_lung_history",
    root=RAW_DIR / "rm1000_lung_history",
    curated_dir=CURATED_DIR,
    seed=RANDOM_SEED
)

summary_table = pd.DataFrame([
    {
        "dataset_id": k,
        "type": v["type"],
        "num": v.get("num_images", v.get("rows_after")),
        "classes": len(v.get("class_counts", v.get("target_counts_after_clean", {}))),
        "notes": "CROPPED" if v.get("used_subfolder") == "CROPPED" else ""
    }
    for k, v in results.items()
])

summary_table

FileNotFoundError: thyroid_recurrence: Nema CSV fajla u /content/drive/MyDrive/Diplomski/Data/raw/thyroid_recurrence